In [1]:
import sqlite3
import pandas as pd

seasons = ["1516","1617","1718","1819","1920","2021","2122","2223","2324","2425"]
frames = []
for s in seasons:
    url = f"https://www.football-data.co.uk/mmz4281/{s}/E0.csv"
    df = pd.read_csv(url, encoding="latin1")
    df["Season"] = s
    frames.append(df[["Season","Date","HomeTeam","AwayTeam","FTHG","FTAG","FTR"]])

matches = pd.concat(frames, ignore_index=True)

conn = sqlite3.connect("pl.db")
matches.to_sql("matches", conn, if_exists="replace", index=False)

pd.read_sql("SELECT Season, COUNT(*) AS games FROM matches GROUP BY Season", conn)

/var/folders/j6/b1k72v8x6pd9mzftsfxbj6lr0000gn/T/ipykernel_2017/2355682270.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Season"] = s


,Season,games
0,1516,380
1,1617,380
2,1718,380
3,1819,380
4,1920,380
5,2021,380
6,2122,380
7,2223,380
8,2324,380
9,2425,380


In [3]:
√pd.read_sql("""
    SELECT HomeTeam,
           COUNT(*) AS games,
           SUM(CASE WHEN FTR = 'H' THEN 1 ELSE 0 END) AS home_wins,
           ROUND(AVG(FTHG), 2) AS avg_goals
    FROM matches
    WHERE Season = '2425'
    GROUP BY HomeTeam
    ORDER BY home_wins DESC
""", conn)

SyntaxError: invalid character '√' (U+221A) (2886226237.py, line 1)

In [4]:
pd.read_sql("""
    SELECT HomeTeam,
           COUNT(*) AS games,
           SUM(CASE WHEN FTR = 'H' THEN 1 ELSE 0 END) AS home_wins,
           ROUND(AVG(FTHG), 2) AS avg_goals
    FROM matches
    WHERE Season = '2425'
    GROUP BY HomeTeam
    ORDER BY home_wins DESC
""", conn)


,HomeTeam,games,home_wins,avg_goals
0,Liverpool,19,14,2.21
1,Man City,19,13,2.26
2,Newcastle,19,12,2.11
3,Chelsea,19,12,1.84
4,Aston Villa,19,11,1.79
5,Arsenal,19,11,1.84
6,Nott'm Forest,19,9,1.37
7,Brentford,19,9,2.11
8,Brighton,19,8,1.58
9,Bournemouth,19,8,1.21


In [5]:
import sqlite3
import pandas as pd

seasons = ["1516","1617","1718","1819","1920","2021","2122","2223","2324","2425","2526"]
frames = []
for s in seasons:
    url = f"https://www.football-data.co.uk/mmz4281/{s}/E0.csv"
    df = pd.read_csv(url, encoding="latin1")
    df = df[["Date","HomeTeam","AwayTeam","FTHG","FTAG","FTR"]].copy()
    df["Season"] = s
    frames.append(df)

matches = pd.concat(frames, ignore_index=True)

conn = sqlite3.connect("pl.db")
matches.to_sql("matches", conn, if_exists="replace", index=False)

pd.read_sql("SELECT Season, COUNT(*) AS games FROM matches GROUP BY Season", conn)

,Season,games
0,1516,380
1,1617,380
2,1718,380
3,1819,380
4,1920,380
5,2021,380
6,2122,380
7,2223,380
8,2324,380
9,2425,380


In [6]:
table = pd.read_sql("""
    WITH team_games AS (
        SELECT Season, HomeTeam AS Team, FTHG AS GF, FTAG AS GA,
               CASE FTR WHEN 'H' THEN 3 WHEN 'D' THEN 1 ELSE 0 END AS Pts
        FROM matches
        UNION ALL
        SELECT Season, AwayTeam, FTAG, FTHG,
               CASE FTR WHEN 'A' THEN 3 WHEN 'D' THEN 1 ELSE 0 END
        FROM matches
    )
    SELECT Season, Team,
           SUM(Pts) AS Pts,
           SUM(GF) - SUM(GA) AS GD,
           RANK() OVER (PARTITION BY Season ORDER BY SUM(Pts) DESC, SUM(GF)-SUM(GA) DESC) AS Position
    FROM team_games
    GROUP BY Season, Team
""", conn)

table[table["Season"] == "2526"].sort_values("Position")

,Season,Team,Pts,GD,Position
200,2526,Arsenal,85,44,1
201,2526,Man City,78,42,2
202,2526,Man United,71,19,3
203,2526,Aston Villa,65,7,4
204,2526,Liverpool,60,10,5
205,2526,Bournemouth,57,4,6
206,2526,Sunderland,54,-6,7
207,2526,Brighton,53,6,8
208,2526,Brentford,53,3,9
209,2526,Chelsea,52,6,10


In [7]:
table.to_sql("season_table", conn, if_exists="replace", index=False)

features = pd.read_sql("""
    WITH t AS (
        SELECT *, DENSE_RANK() OVER (ORDER BY Season) AS season_idx
        FROM season_table
    ),
    lagged AS (
        SELECT *,
               LAG(season_idx) OVER w AS prev_idx,
               LAG(Pts)        OVER w AS prev_pts,
               LAG(GD)         OVER w AS prev_gd,
               LAG(Position)   OVER w AS prev_pos,
               AVG(Pts) OVER (PARTITION BY Team ORDER BY Season
                              ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING) AS avg_pts_3
        FROM t
        WINDOW w AS (PARTITION BY Team ORDER BY Season)
    )
    SELECT Season, Team, Position, Pts,
           prev_pts, prev_gd, prev_pos, avg_pts_3,
           CASE WHEN prev_idx IS NULL OR prev_idx != season_idx - 1
                THEN 1 ELSE 0 END AS promoted
    FROM lagged
""", conn)

features[features["Season"] == "2526"].sort_values("Position")

,Season,Team,Position,Pts,prev_pts,prev_gd,prev_pos,avg_pts_3,promoted
10,2526,Arsenal,1,85,74.0,35.0,2.0,82.333333,0
129,2526,Man City,2,78,71.0,28.0,3.0,83.666667,0
140,2526,Man United,3,71,42.0,-10.0,15.0,59.000000,0
18,2526,Aston Villa,4,65,66.0,7.0,6.0,65.000000,0
117,2526,Liverpool,5,60,84.0,45.0,1.0,77.666667,0
27,2526,Bournemouth,6,57,56.0,12.0,9.0,47.666667,0
176,2526,Sunderland,7,54,24.0,-40.0,20.0,31.500000,1
41,2526,Brighton,8,53,61.0,7.0,8.0,57.000000,0
32,2526,Brentford,9,53,56.0,9.0,10.0,51.333333,0
61,2526,Chelsea,10,52,69.0,21.0,4.0,58.666667,0


In [8]:
%pip install scikit-learn -q


Note: you may need to restart the kernel to use updated packages.


In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

df = features[features["Season"] != "1516"].copy()   # first season has no history
cols = ["prev_pts", "prev_gd", "avg_pts_3"]
df.loc[df["promoted"] == 1, cols] = 0                # wipe stale stats for promoted teams
X = cols + ["promoted"]

train = df[df["Season"] <= "2324"]
test = df[df["Season"].isin(["2425", "2526"])].copy()

model = LinearRegression().fit(train[X], train["Pts"])
test["pred_pts"] = model.predict(test[X])
test["pred_pos"] = test.groupby("Season")["pred_pts"].rank(ascending=False, method="first")
test["base_pos"] = test.groupby("Season")["prev_pts"].rank(ascending=False, method="first")

for s, g in test.groupby("Season"):
    print(s, "| model MAE:", round(mean_absolute_error(g["Position"], g["pred_pos"]), 2),
             "| baseline MAE:", round(mean_absolute_error(g["Position"], g["base_pos"]), 2))

2425 | model MAE: 2.8 | baseline MAE: 3.3
2526 | model MAE: 3.8 | baseline MAE: 3.7


In [10]:
test_seasons = [s for s in sorted(df["Season"].unique()) if s >= "1920"]
rows = []

for s in test_seasons:
    tr = df[df["Season"] < s]                 # only past seasons
    te = df[df["Season"] == s].copy()         # the season we predict

    m = LinearRegression().fit(tr[X], tr["Pts"])
    te["pred_pts"] = m.predict(te[X])
    te["pred_pos"] = te["pred_pts"].rank(ascending=False, method="first")
    te["base_pos"] = te["prev_pts"].rank(ascending=False, method="first")

    rows.append({
        "Season": s,
        "train_seasons": tr["Season"].nunique(),
        "model_MAE": round(mean_absolute_error(te["Position"], te["pred_pos"]), 2),
        "baseline_MAE": round(mean_absolute_error(te["Position"], te["base_pos"]), 2),
    })

results = pd.DataFrame(rows)
results["model_better"] = results["model_MAE"] < results["baseline_MAE"]

print("Avg model MAE:   ", round(results["model_MAE"].mean(), 2))
print("Avg baseline MAE:", round(results["baseline_MAE"].mean(), 2))
print("Model wins", results["model_better"].sum(), "of", len(results), "seasons")
results

Avg model MAE:    3.42
Avg baseline MAE: 3.35
Model wins 3 of 7 seasons


,Season,train_seasons,model_MAE,baseline_MAE,model_better
0,1920,3,3.80,3.30,False
1,2021,4,3.35,3.25,False
2,2122,5,2.50,2.70,True
3,2223,6,5.10,4.40,False
4,2324,7,2.60,2.80,True
5,2425,8,2.80,3.30,True
6,2526,9,3.80,3.70,False


In [11]:
frames = []
for s in seasons:
    url = f"https://www.football-data.co.uk/mmz4281/{s}/E0.csv"
    d = pd.read_csv(url, encoding="latin1")
    d = d[["Date","HomeTeam","AwayTeam","FTHG","FTAG","FTR","HST","AST"]].copy()
    d["Season"] = s
    frames.append(d)

matches = pd.concat(frames, ignore_index=True)
matches.to_sql("matches", conn, if_exists="replace", index=False)

4180

In [12]:
features2 = pd.read_sql("""
    WITH team_games AS (
        SELECT Season, HomeTeam AS Team, FTHG AS GF, FTAG AS GA, HST AS SOTF, AST AS SOTA,
               CASE FTR WHEN 'H' THEN 3 WHEN 'D' THEN 1 ELSE 0 END AS Pts
        FROM matches
        UNION ALL
        SELECT Season, AwayTeam, FTAG, FTHG, AST, HST,
               CASE FTR WHEN 'A' THEN 3 WHEN 'D' THEN 1 ELSE 0 END
        FROM matches
    ),
    st AS (
        SELECT Season, Team, SUM(Pts) AS Pts, SUM(GF) - SUM(GA) AS GD,
               1.0 * SUM(SOTF) / (SUM(SOTF) + SUM(SOTA)) AS sot_share,
               RANK() OVER (PARTITION BY Season ORDER BY SUM(Pts) DESC, SUM(GF)-SUM(GA) DESC) AS Position
        FROM team_games GROUP BY Season, Team
    ),
    t AS (SELECT *, DENSE_RANK() OVER (ORDER BY Season) AS season_idx FROM st),
    lagged AS (
        SELECT *, LAG(season_idx) OVER w AS prev_idx,
               LAG(Pts) OVER w AS prev_pts, LAG(GD) OVER w AS prev_gd,
               LAG(sot_share) OVER w AS prev_sot_share,
               AVG(Pts) OVER (PARTITION BY Team ORDER BY Season
                              ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING) AS avg_pts_3
        FROM t WINDOW w AS (PARTITION BY Team ORDER BY Season)
    )
    SELECT Season, Team, Position, Pts, prev_pts, prev_gd, prev_sot_share, avg_pts_3,
           CASE WHEN prev_idx IS NULL OR prev_idx != season_idx - 1 THEN 1 ELSE 0 END AS promoted
    FROM lagged
""", conn)

df2 = features2[features2["Season"] != "1516"].copy()
df2.loc[df2["promoted"] == 1, ["prev_pts","prev_gd","prev_sot_share","avg_pts_3"]] = 0

In [13]:
def walk_forward(data, X):
    maes = []
    for s in [x for x in sorted(data["Season"].unique()) if x >= "1920"]:
        tr, te = data[data["Season"] < s], data[data["Season"] == s].copy()
        m = LinearRegression().fit(tr[X], tr["Pts"])
        te["pred_pos"] = pd.Series(m.predict(te[X]), index=te.index).rank(ascending=False, method="first")
        maes.append(mean_absolute_error(te["Position"], te["pred_pos"]))
    return round(sum(maes) / len(maes), 2)

base_X = ["prev_pts", "prev_gd", "avg_pts_3", "promoted"]
shots_X = base_X + ["prev_sot_share"]

print("Goals/points features: ", walk_forward(df2, base_X))
print("+ shots on target share:", walk_forward(df2, shots_X))

Goals/points features:  3.42
+ shots on target share: 3.14


In [14]:
def wf_by_season(data, X):
    out = {}
    for s in [x for x in sorted(data["Season"].unique()) if x >= "1920"]:
        tr, te = data[data["Season"] < s], data[data["Season"] == s].copy()
        m = LinearRegression().fit(tr[X], tr["Pts"])
        te["pred_pos"] = pd.Series(m.predict(te[X]), index=te.index).rank(ascending=False, method="first")
        out[s] = round(mean_absolute_error(te["Position"], te["pred_pos"]), 2)
    return pd.Series(out)

comp = pd.DataFrame({"base": wf_by_season(df2, base_X), "shots": wf_by_season(df2, shots_X)})
comp["shots_better"] = comp["shots"] < comp["base"]
comp

,base,shots,shots_better
1920,3.80,3.50,True
2021,3.35,2.95,True
2122,2.50,2.40,True
2223,5.10,4.40,True
2324,2.60,2.30,True
2425,2.80,2.50,True
2526,3.80,3.90,False


In [15]:
stats = pd.read_sql("""
    WITH team_games AS (
        SELECT Season, HomeTeam AS Team, FTHG AS GF, FTAG AS GA, HST AS SOTF, AST AS SOTA,
               CASE FTR WHEN 'H' THEN 3 WHEN 'D' THEN 1 ELSE 0 END AS Pts
        FROM matches
        UNION ALL
        SELECT Season, AwayTeam, FTAG, FTHG, AST, HST,
               CASE FTR WHEN 'A' THEN 3 WHEN 'D' THEN 1 ELSE 0 END
        FROM matches
    )
    SELECT Season, Team, SUM(Pts) AS Pts, SUM(GF) - SUM(GA) AS GD,
           1.0 * SUM(SOTF) / (SUM(SOTF) + SUM(SOTA)) AS sot_share
    FROM team_games GROUP BY Season, Team
""", conn).sort_values("Season")

last = stats[stats["Season"] == "2526"].set_index("Team")
avg3 = stats.groupby("Team").tail(3).groupby("Team")["Pts"].mean()

relegated = ["West Ham", "Burnley", "Wolves"]
promoted_in = ["Coventry", "Ipswich", "Hull"]

rows = [{"Team": t, "prev_pts": last.loc[t, "Pts"], "prev_gd": last.loc[t, "GD"],
         "prev_sot_share": last.loc[t, "sot_share"], "avg_pts_3": avg3[t], "promoted": 0}
        for t in last.index if t not in relegated]
rows += [{"Team": t, "prev_pts": 0, "prev_gd": 0, "prev_sot_share": 0,
          "avg_pts_3": 0, "promoted": 1} for t in promoted_in]
new = pd.DataFrame(rows)

final_model = LinearRegression().fit(df2[shots_X], df2["Pts"])   # trained on all seasons
new["pred_pts"] = final_model.predict(new[shots_X]).round(1)
new["pred_pos"] = new["pred_pts"].rank(ascending=False, method="first").astype(int)
new.sort_values("pred_pos")[["pred_pos", "Team", "pred_pts"]]

,pred_pos,Team,pred_pts
15,1,Arsenal,77.7
1,2,Man City,73.9
0,3,Liverpool,63.7
2,4,Man United,61.5
11,5,Chelsea,58.5
9,6,Brighton,57.9
7,7,Newcastle,56.4
14,8,Aston Villa,55.6
13,9,Bournemouth,54.5
4,10,Nott'm Forest,52.4


In [16]:
champ_frames = []
for s in seasons:
    url = f"https://www.football-data.co.uk/mmz4281/{s}/E1.csv"
    d = pd.read_csv(url, encoding="latin1").dropna(subset=["HomeTeam"])
    d = d[["HomeTeam","AwayTeam","FTHG","FTAG","FTR","HST","AST"]].copy()
    d["Season"] = s
    champ_frames.append(d)

pd.concat(champ_frames, ignore_index=True).to_sql("champ", conn, if_exists="replace", index=False)

6072

In [17]:
champ_stats = pd.read_sql("""
    WITH tg AS (
        SELECT Season, HomeTeam AS Team, HST AS SOTF, AST AS SOTA,
               CASE FTR WHEN 'H' THEN 3 WHEN 'D' THEN 1 ELSE 0 END AS Pts
        FROM champ
        UNION ALL
        SELECT Season, AwayTeam, AST, HST,
               CASE FTR WHEN 'A' THEN 3 WHEN 'D' THEN 1 ELSE 0 END
        FROM champ
    )
    SELECT Season, Team,
           1.0 * SUM(Pts) / COUNT(*) AS champ_ppg,
           1.0 * SUM(SOTF) / (SUM(SOTF) + SUM(SOTA)) AS champ_sot_share
    FROM tg GROUP BY Season, Team
""", conn)

# Championship 2024-25 stats become features for PL 2025-26, etc.
next_season = {seasons[i]: seasons[i+1] for i in range(len(seasons) - 1)}
next_season["2526"] = "2627"
champ_stats["Season"] = champ_stats["Season"].map(next_season)

df3 = df2.merge(champ_stats, on=["Season", "Team"], how="left")
df3.loc[df3["promoted"] == 0, ["champ_ppg", "champ_sot_share"]] = 0

print("Promoted rows missing Championship data:", df3.loc[df3["promoted"] == 1, "champ_ppg"].isna().sum())

Promoted rows missing Championship data: 0


In [18]:
champ_X = shots_X + ["champ_ppg", "champ_sot_share"]
print("Shots model:        ", walk_forward(df3, shots_X))
print("+ Championship info:", walk_forward(df3, champ_X))

Shots model:         3.14
+ Championship info: 3.16


In [19]:
stats = pd.read_sql("""
    WITH team_games AS (
        SELECT Season, HomeTeam AS Team, FTHG AS GF, FTAG AS GA, HST AS SOTF, AST AS SOTA,
               CASE FTR WHEN 'H' THEN 3 WHEN 'D' THEN 1 ELSE 0 END AS Pts
        FROM matches
        UNION ALL
        SELECT Season, AwayTeam, FTAG, FTHG, AST, HST,
               CASE FTR WHEN 'A' THEN 3 WHEN 'D' THEN 1 ELSE 0 END
        FROM matches
    )
    SELECT Season, Team, SUM(Pts) AS Pts, SUM(GF) - SUM(GA) AS GD,
           1.0 * SUM(SOTF) / (SUM(SOTF) + SUM(SOTA)) AS sot_share
    FROM team_games GROUP BY Season, Team
""", conn).sort_values("Season")

last = stats[stats["Season"] == "2526"].set_index("Team")
avg3 = stats.groupby("Team").tail(3).groupby("Team")["Pts"].mean()

relegated = ["West Ham", "Burnley", "Wolves"]
promoted_in = ["Coventry", "Ipswich", "Hull"]

rows = [{"Team": t, "prev_pts": last.loc[t, "Pts"], "prev_gd": last.loc[t, "GD"],
         "prev_sot_share": last.loc[t, "sot_share"], "avg_pts_3": avg3[t], "promoted": 0}
        for t in last.index if t not in relegated]
rows += [{"Team": t, "prev_pts": 0, "prev_gd": 0, "prev_sot_share": 0,
          "avg_pts_3": 0, "promoted": 1} for t in promoted_in]
new = pd.DataFrame(rows)

final_model = LinearRegression().fit(df2[shots_X], df2["Pts"])
new["pred_pts"] = final_model.predict(new[shots_X]).round(1)
new["pred_pos"] = new["pred_pts"].rank(ascending=False, method="first").astype(int)
new.sort_values("pred_pos")[["pred_pos", "Team", "pred_pts"]]

,pred_pos,Team,pred_pts
15,1,Arsenal,77.7
1,2,Man City,73.9
0,3,Liverpool,63.7
2,4,Man United,61.5
11,5,Chelsea,58.5
9,6,Brighton,57.9
7,7,Newcastle,56.4
14,8,Aston Villa,55.6
13,9,Bournemouth,54.5
4,10,Nott'm Forest,52.4


In [20]:
rows = []
for s in [x for x in sorted(df2["Season"].unique()) if x >= "1920"]:
    tr, te = df2[df2["Season"] < s], df2[df2["Season"] == s].copy()
    m = LinearRegression().fit(tr[shots_X], tr["Pts"])
    te["pred_pos"] = pd.Series(m.predict(te[shots_X]), index=te.index).rank(ascending=False, method="first")
    te["base_pos"] = te["prev_pts"].rank(ascending=False, method="first")
    rows.append({"Season": s,
                 "model_MAE": round(mean_absolute_error(te["Position"], te["pred_pos"]), 2),
                 "baseline_MAE": round(mean_absolute_error(te["Position"], te["base_pos"]), 2)})

results = pd.DataFrame(rows)
results["model_better"] = results["model_MAE"] < results["baseline_MAE"]
print("Avg model MAE:   ", round(results["model_MAE"].mean(), 2))
print("Avg baseline MAE:", round(results["baseline_MAE"].mean(), 2))
print("Model wins", results["model_better"].sum(), "of", len(results), "seasons")
results

Avg model MAE:    3.14
Avg baseline MAE: 3.35
Model wins 4 of 7 seasons


,Season,model_MAE,baseline_MAE,model_better
0,1920,3.50,3.30,False
1,2021,2.95,3.25,True
2,2122,2.40,2.70,True
3,2223,4.40,4.40,False
4,2324,2.30,2.80,True
5,2425,2.50,3.30,True
6,2526,3.90,3.70,False


In [21]:
features2 = pd.read_sql("""
    WITH team_games AS (
        SELECT Season, HomeTeam AS Team, FTHG AS GF, FTAG AS GA, HST AS SOTF, AST AS SOTA,
               CASE FTR WHEN 'H' THEN 3 WHEN 'D' THEN 1 ELSE 0 END AS Pts
        FROM matches
        UNION ALL
        SELECT Season, AwayTeam, FTAG, FTHG, AST, HST,
               CASE FTR WHEN 'A' THEN 3 WHEN 'D' THEN 1 ELSE 0 END
        FROM matches
    ),
    st AS (
        SELECT Season, Team, SUM(Pts) AS Pts, SUM(GF) - SUM(GA) AS GD,
               1.0 * SUM(SOTF) / (SUM(SOTF) + SUM(SOTA)) AS sot_share,
               RANK() OVER (PARTITION BY Season ORDER BY SUM(Pts) DESC, SUM(GF)-SUM(GA) DESC) AS Position
        FROM team_games GROUP BY Season, Team
    ),
    t AS (SELECT *, DENSE_RANK() OVER (ORDER BY Season) AS season_idx FROM st),
    lagged AS (
        SELECT *, LAG(season_idx) OVER w AS prev_idx,
               LAG(Pts) OVER w AS prev_pts, LAG(GD) OVER w AS prev_gd,
               LAG(sot_share) OVER w AS prev_sot_share,
               AVG(Pts) OVER (PARTITION BY Team ORDER BY season_idx
                              RANGE BETWEEN 3 PRECEDING AND 1 PRECEDING) AS avg_pts_3
        FROM t WINDOW w AS (PARTITION BY Team ORDER BY season_idx)
    )
    SELECT Season, Team, Position, Pts, prev_pts, prev_gd, prev_sot_share, avg_pts_3,
           CASE WHEN prev_idx IS NULL OR prev_idx != season_idx - 1 THEN 1 ELSE 0 END AS promoted
    FROM lagged
""", conn)

df2 = features2[features2["Season"] != "1516"].copy()
df2.loc[df2["promoted"] == 1, ["prev_pts", "prev_gd", "prev_sot_share", "avg_pts_3"]] = 0
df2["avg_pts_3"] = df2["avg_pts_3"].fillna(df2["prev_pts"])

base_X = ["prev_pts", "prev_gd", "avg_pts_3", "promoted"]
shots_X = base_X + ["prev_sot_share"]


In [22]:
stats = pd.read_sql("""
    WITH team_games AS (
        SELECT Season, HomeTeam AS Team, FTHG AS GF, FTAG AS GA, HST AS SOTF, AST AS SOTA,
               CASE FTR WHEN 'H' THEN 3 WHEN 'D' THEN 1 ELSE 0 END AS Pts
        FROM matches
        UNION ALL
        SELECT Season, AwayTeam, FTAG, FTHG, AST, HST,
               CASE FTR WHEN 'A' THEN 3 WHEN 'D' THEN 1 ELSE 0 END
        FROM matches
    )
    SELECT Season, Team, SUM(Pts) AS Pts, SUM(GF) - SUM(GA) AS GD,
           1.0 * SUM(SOTF) / (SUM(SOTF) + SUM(SOTA)) AS sot_share
    FROM team_games GROUP BY Season, Team
""", conn)

last = stats[stats["Season"] == "2526"].set_index("Team")
avg3 = stats[stats["Season"].isin(["2324", "2425", "2526"])].groupby("Team")["Pts"].mean()

relegated = ["West Ham", "Burnley", "Wolves"]
promoted_in = ["Coventry", "Ipswich", "Hull"]

rows = [{"Team": t, "prev_pts": last.loc[t, "Pts"], "prev_gd": last.loc[t, "GD"],
         "prev_sot_share": last.loc[t, "sot_share"], "avg_pts_3": avg3[t], "promoted": 0}
        for t in last.index if t not in relegated]
rows += [{"Team": t, "prev_pts": 0, "prev_gd": 0, "prev_sot_share": 0,
          "avg_pts_3": 0, "promoted": 1} for t in promoted_in]
new = pd.DataFrame(rows)

final_model = LinearRegression().fit(df2[shots_X], df2["Pts"])
new["pred_pts"] = final_model.predict(new[shots_X]).round(1)
new["pred_pos"] = new["pred_pts"].rank(ascending=False, method="first").astype(int)
new.sort_values("pred_pos")[["pred_pos", "Team", "pred_pts"]]

,pred_pos,Team,pred_pts
0,1,Arsenal,77.9
11,2,Man City,74.2
10,3,Liverpool,64.4
12,4,Man United,61.1
5,5,Chelsea,58.7
4,6,Brighton,57.8
13,7,Newcastle,56.6
1,8,Aston Villa,55.9
2,9,Bournemouth,54.3
14,10,Nott'm Forest,52.3


In [23]:
new[new["Team"] == "Sunderland"]

,Team,prev_pts,prev_gd,prev_sot_share,avg_pts_3,promoted,pred_pts,pred_pos
15,Sunderland,54,-6,0.430034,54.0,0,45.7,17


In [24]:
rows = []
for s in [x for x in sorted(df2["Season"].unique()) if x >= "1920"]:
    tr, te = df2[df2["Season"] < s], df2[df2["Season"] == s].copy()
    m = LinearRegression().fit(tr[shots_X], tr["Pts"])
    te["pred_pos"] = pd.Series(m.predict(te[shots_X]), index=te.index).rank(ascending=False, method="first")
    te["base_pos"] = te["prev_pts"].rank(ascending=False, method="first")
    rows.append({"Season": s,
                 "model_MAE": round(mean_absolute_error(te["Position"], te["pred_pos"]), 2),
                 "baseline_MAE": round(mean_absolute_error(te["Position"], te["base_pos"]), 2)})

results = pd.DataFrame(rows)
results["model_better"] = results["model_MAE"] < results["baseline_MAE"]
print("Avg model MAE:   ", round(results["model_MAE"].mean(), 2))
print("Avg baseline MAE:", round(results["baseline_MAE"].mean(), 2))
print("Model wins", results["model_better"].sum(), "of", len(results), "seasons")
results

Avg model MAE:    3.08
Avg baseline MAE: 3.35
Model wins 5 of 7 seasons


,Season,model_MAE,baseline_MAE,model_better
0,1920,3.50,3.30,False
1,2021,2.75,3.25,True
2,2122,2.40,2.70,True
3,2223,4.30,4.40,True
4,2324,2.30,2.80,True
5,2425,2.50,3.30,True
6,2526,3.80,3.70,False
